# Jon2 AI Arabic training
Offline training from the three explicitly approved original Arabic dialogue batches. Research use only.

In [ ]:
import shutil, torch
from pathlib import Path
from argparse import Namespace
INPUT_ROOT=Path('/kaggle/input')
print('Mounted Kaggle inputs:', [str(p) for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else 'MISSING')
source_matches=list(INPUT_ROOT.glob('**/training_data/original_dialogue_ar_batch_01.txt')) if INPUT_ROOT.exists() else []
if not source_matches:
    raise FileNotFoundError('Approved batch 01 was not found in mounted Kaggle inputs. Verify dataset access and dataset_sources. Mounted inputs: '+str([str(p) for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else []))
INPUT=source_matches[0].parent.parent
print('Using approved dataset root:',INPUT)
ROOT=Path('/kaggle/working/jon2-engine')
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
for name in ('jon2_core.py','train_jon2.py','tokenizer.py','source_approval.py'): shutil.copy2(INPUT/name,ROOT/name)
DATA=Path('/kaggle/working/jon2-data')
if DATA.exists(): shutil.rmtree(DATA)
shutil.copytree(INPUT/'training_data',DATA)
import sys
sys.path.insert(0,str(ROOT))
from source_approval import validate_source_approvals
from tokenizer import ByteBPETokenizer
validate_source_approvals(DATA,DATA/'source_approvals.json')
files=sorted(DATA.glob('*.txt'))
assert len(files)==3
corpus='\n\n'.join(p.read_bytes().decode('utf-8',errors='strict') for p in files)
assert '\ufffd' not in corpus
tok=ByteBPETokenizer.train(corpus[:50000],vocab_size=512)
assert tok.decode(tok.encode(corpus[:12000]))==corpus[:12000]
print('UTF-8 round trip passed. Corpus characters:',len(corpus))
from train_jon2 import train
args=Namespace(data=str(DATA),approval_manifest=str(DATA/'source_approvals.json'),output='/kaggle/working/jon2-checkpoints/jon2-small.pt',last_checkpoint='/kaggle/working/jon2-checkpoints/jon2-last.pt',resume='',steps=1200,batch_size=8,context=96,width=96,heads=4,layers=3,dropout=0.1,learning_rate=3e-4,weight_decay=0.1,warmup_steps=20,validation_fraction=0.1,eval_steps=8,eval_every=100,patience=8,min_delta=0.001,grad_clip=1.0,minimum_chars=100000,tokenizer_vocab_size=768,tokenizer_min_frequency=2,seed=42,no_amp=False,keep_last=True)
train(args)


In [ ]:
import inspect, refrom jon2_core import load_model,generateBEST='/kaggle/working/jon2-checkpoints/jon2-small.pt'model,tokenizer,device=load_model(BEST)state=torch.load(BEST,map_location='cpu',weights_only=False)print('Training steps:',state.get('training_steps'),'validation loss:',state.get('best_validation_loss'))print('Loaded generate signature:',inspect.signature(generate))questions=['ما الفرق بين البيانات والمعلومات؟','احسب 15 بالمئة من 200.','كيف أتحقق من صحة خبر؟','ما وظيفة نظام التشغيل؟','كيف أتعامل مع خطأ في برنامج؟','لماذا لا يثبت الارتباط السببية؟','ما الفرق بين الكتلة والوزن؟','كيف أحمي كلمة المرور؟']quality=[]for question in questions: prompt='المستخدم: '+question+'\nJon2:' options={'max_new_chars':180,'temperature':0.8,'stop_sequences':[]} supported=inspect.signature(generate).parameters if 'min_new_tokens' in supported: options['min_new_tokens']=24 if 'do_sample' in supported: options['do_sample']=True if 'repetition_penalty' in supported: options['repetition_penalty']=1.15 if 'no_repeat_ngram_size' in supported: options['no_repeat_ngram_size']=3 if 'top_k' in supported: options['top_k']=40 result=generate(model,tokenizer,prompt,device,**options) answer=result[len(prompt):].strip() if result.startswith(prompt) else result.strip() words=re.findall(r'[\u0600-\u06FF]{2,}',answer) unique_ratio=(len(set(words))/len(words)) if words else 0.0 adjacent_repeats=sum(1 for a,b in zip(words,words[1:]) if a==b) repeat_ratio=adjacent_repeats/max(1,len(words)-1) passed=len(words)>=8 and unique_ratio>=0.45 and repeat_ratio<=0.25 and '\ufffd' not in answer quality.append(passed) print('\\nسؤال:',question,'\\nإجابة:',repr(answer),'\\nمؤشرات: كلمات=',len(words),'تنوع=',round(unique_ratio,2),'تكرار متجاور=',round(repeat_ratio,2),'الحالة=',('مقبول مبدئيًا' if passed else 'ضعيف'),flush=True)print('Arabic heuristic checks:',sum(quality),'/',len(quality))if all(quality): print('QUALITY STATUS: HEURISTIC PASS ONLY — human factual review is still required.')else: print('QUALITY STATUS: NOT READY — checkpoint is preserved for further research, but must NOT be deployed as a chat assistant.')

UTF-8 integrity does not establish factual accuracy or ChatGPT-level capability. Checkpoint storage remains private.